In [79]:
from selenium import webdriver
from selenium.webdriver import Chrome, ChromeOptions
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import  WebDriverWait
import pandas as pd 
from selenium.webdriver.common.action_chains import ActionChains
import time, threading, os, requests, ast,json,re
from datetime import datetime

In [80]:
def GetAuctionDetails(driver):
    wait = WebDriverWait(driver, 10)

    try:
        
        
        
        auction_name_el = wait.until(
            EC.presence_of_element_located((By.XPATH, "//h1[@class='cc-auction-overview__title']"))
        )

        auction_time_el = wait.until(
            EC.presence_of_element_located((By.XPATH, "//div[@class='cc-auction-overview__detail cc-auction-overview__when']//dd"))
        )

        center_name_el = wait.until(
            EC.presence_of_element_located((By.XPATH, "//div[@class='u-line-height-1']"))
        )

        auction_name = auction_name_el.text.strip()
        auction_time_raw = auction_time_el.text.strip()  
        center_name = center_name_el.text.strip()
        auction_time_clean = re.sub(r"^\w+\s", "", auction_time_raw) 
        auction_time_clean = auction_time_clean.replace(" at ", " ")     
        current_year = datetime.now().year
        dt = datetime.strptime(f"{auction_time_clean} {current_year}", "%d %B %I:%M%p %Y")

        auction_date = dt.strftime("%Y-%m-%d")
        auction_time = dt.strftime("%H:%M")

        auction_data = {
            "auction_name": auction_name,
            "auction_date": auction_date,
            "auction_time": auction_time,
            "center_name": center_name
        }

        # Save JSON
        with open("database.json", "w", encoding="utf-8") as f:
            json.dump(auction_data, f, indent=4)

        print("✅ Auction details saved successfully")
        print(json.dumps(auction_data, indent=4))

        return auction_data

    except Exception as e:
        print("❌ Error in GetAuctionDetails:", e)
        return None


def scarpe(id,name="finance-and-dealership-car-auction-"):
    path = f"https://www.wilsonsauctions.com/auctions/{name}{id}"
    options = ChromeOptions()
    options.headless = True
    service = Service(ChromeDriverManager().install())
    driver = Chrome(service=service, options=options)
    driver.get(path)
    driver.maximize_window()

    wait = WebDriverWait(driver, 10)
    try:
        cookie_btn = wait.until(EC.element_to_be_clickable((By.ID, 'cookiescript_accept')))
        cookie_btn.click()
  
    except:
        print("No cookies button found")


    try:
        login_btn = wait.until(EC.element_to_be_clickable((By.ID, 'gtm-sign-in-sign-up')))
        login_btn.click()
    
    except:
        print("Login button not found")


    try:
        username_input = wait.until(EC.presence_of_element_located((By.ID, 'email')))
        username_input.send_keys("fourbrotherstrading@icloud.com")
        next_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@class='c-button']")))
        next_btn.click()

    except:
        print("Username input not found")

    try:
        password_input = wait.until(EC.visibility_of_element_located((By.ID, 'password')))
        password_input.clear()
        password_input.send_keys("Muhssan7865@")  
        login_btn2 = wait.until(
            EC.element_to_be_clickable((By.XPATH, "//button[@class='c-button' and @data-label='log-in']"))
        )
        login_btn2.click()

    except Exception as e:
        print("Password input not found or error:", e)
    GetAuctionDetails(driver)
    try:
        first_lot = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//ul[@class='cc-cards +list +results u-marg-top u-pad-top']//li[1]//a")
            )
        )
        lot_url = first_lot.get_attribute("href")
        print(f"Opening first lot: {lot_url}")
        driver.execute_script("arguments[0].click();", first_lot)
    except Exception as e:
        print("Could not open first lot:", e)
        
    folder_name = "html"
    os.makedirs(folder_name, exist_ok=True)

    lot_number = 1

    while True:
        try:
            get_Reg = wait.until(
                EC.presence_of_element_located(
                    (By.XPATH, "//div[@class='cc-reg-plate__end']//span")
                )
            )
            reg_number = get_Reg.text.strip()
            print(f"🚗 Registration Number: {reg_number}")

            # 🔹 OPEN INSPECTION TAB
            try:
                inspect_tab = wait.until(
                    EC.element_to_be_clickable((
                        By.XPATH,
                        "//button[contains(., 'Inspection')] | //a[contains(., 'Inspection')]"
                    ))
                )
                driver.execute_script("arguments[0].click();", inspect_tab)
                time.sleep(2)
            except:
                pass

            # 🔹 EXPAND ALL SECTIONS (if any)
            try:
                expands = driver.find_elements(
                    By.XPATH, "//button[contains(@class,'accordion')]"
                )
                for e in expands:
                    driver.execute_script("arguments[0].click();", e)
                    time.sleep(0.3)
            except:
                pass

            # 🔹 SAVE FULL HTML
            with open(f"{folder_name}/{reg_number}.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)

            print(f"✅ Saved inspection HTML: {reg_number}.html")

            # 🔹 NEXT LOT

            next_btn = wait.until(
                EC.element_to_be_clickable((
                    By.XPATH,
                    "//button[.//span[contains(normalize-space(), 'Next')]]"
                ))
            )
            print(next_btn)
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(2)

            lot_number += 1

        except Exception as e:
            print("🛑 No more lots:", e)
            break

scarpe(1732,"main-agent-car-commercial-and-ex-fire-service-equipment-auction-")
        

✅ Auction details saved successfully
{
    "auction_name": "Main Agent Car, Commercial and ex-Fire Service Equipment Auction",
    "auction_date": "2025-12-18",
    "auction_time": "14:00",
    "center_name": "Wilsons Auctions Oxford"
}
Opening first lot: https://www.wilsonsauctions.com/auctions/main-agent-car-commercial-and-ex-fire-service-equipment-auction-1732/lots/209510
🚗 Registration Number: HN14PUA
✅ Saved inspection HTML: HN14PUA.html
<selenium.webdriver.remote.webelement.WebElement (session="88e177e36aef7a2f79bbd4e5f9734462", element="f.A01C85158FA3E77E4B422DEA142343E9.d.0CE130589DC07D5EDF3484CBD046B1BF.e.127")>
🚗 Registration Number: WP16FFH
✅ Saved inspection HTML: WP16FFH.html
<selenium.webdriver.remote.webelement.WebElement (session="88e177e36aef7a2f79bbd4e5f9734462", element="f.A01C85158FA3E77E4B422DEA142343E9.d.0CE130589DC07D5EDF3484CBD046B1BF.e.131")>
🚗 Registration Number: KX66VZC
✅ Saved inspection HTML: KX66VZC.html
<selenium.webdriver.remote.webelement.WebElement (s